# **P8 - Kriging Geográfico (2D)**

*En esta parte de la práctica, implementaremos un modelo simple de Kriging sobre los datos de <br>
calidad del aire en Madrid, proporcionados por el Ayuntamiento en su [portal
de datos abiertos](https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?vgnextoid=9e42c176313eb410VgnVCM1000000b205a0aRCRD&vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default).*

Como de costumbre, primero importamos las librerías y módulos necesarios.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from PIL import Image
from pyproj import Transformer
from scipy.spatial import ConvexHull, Delaunay
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    WhiteKernel,
    ConstantKernel,
    RationalQuadratic,
)

# descomenta %matplotlib qt si prefieres plots interactivos
%matplotlib inline
# %matplotlib qt


## 0. Setup y Funciones Auxiliares

### 0.1 Transformación de coordenadas

Las coordenadas geográficas que emplearemos vienen dadas en grados (*latitud* y *longitud*). <br>
Por tanto, necesitamos una forma de transformarlas (o *proyectarlas*) a unidades métricas (p. ej. metros), <br>
por ser éstas más adecuadas en el cálculo de distancias necesarias en **Kriging**.

Para estas transformaciones, usaremos la librería **PyProj** [[*GH*](https://github.com/pyproj4/pyproj) | [*Docs*](https://pyproj4.github.io/pyproj/stable/)].

<div class="alert alert-info">
&#9432; <strong>INFO</strong> <br>
<em> Estas transformaciones se realizan usando los siguientes
<strong>sistemas de referencia de coordenadas</strong> (<strong>CRS</strong>): <br>
</em>

* <em>EPSG:<strong>4326</strong></em> (WGS84) <br>
por ser el CRS apropiado para coordenadas de latitud y longitud (representadas en grados)
[<a href="https://epsg.io/?q=4326">link</a>].

* <em>EPSG:<strong>32630</strong></em> (UTM, zona 30N) <br>
CRS empleado para expresar en metros las coordenadas de Madrid, proyectándolas a un plano 2D local. <br>
La zona UTM 30N cubre la longitud de Madrid y evita trabajar con distancias angulares.

</div>


In [2]:
transformer_longlat_xy = Transformer.from_crs(4326, 32630, always_xy=True)


def from_longlat_to_xy(long, lat):
    """Transforma coordenadas de longitud y latitud a coordenadas x, y en CRS 32630
    (UTM, zona 30)

    Args:
        long: (n,) array o float con longitud (en grados)
        lat: (n,) array o float con latitud (en grados)

    Returns:
        x: (n,) array o float con coordenadas x en CRS 32630
        y: (n,) array o float con coordenadas y en CRS 32630
    """
    return transformer_longlat_xy.transform(long, lat)


def from_xy_to_longlat(x, y):
    """Transforma coordenadas x, y en CRS 32630 (UTM, zona 30) a coordenadas de longitud
    y latitud

    Args:
        x: (n,) array o float con coordenadas x en CRS 32630
        y: (n,) array o float con coordenadas y en CRS 32630

    Returns:
        long: (n,) array o float con longitud (en grados)
        lat: (n,) array o float con latitud (en grados)
    """
    return transformer_longlat_xy.transform(x, y, direction="inverse")

### 0.2 Carga de datos

En primer lugar, definimos  las **coordenadas geográficas** (latitud y longitud) <br>
**de las estaciones** de medición de calidad del aire en Madrid (ver [enlace]((https://datos.madrid.es/FWProjects/egob/Catalogo/MedioAmbiente/Aire/Ficheros/Interprete_ficheros_%20calidad_%20del_%20aire_global.pdf)), Anexo 1).


In [3]:
# Coordinates for the stations
station_coordinates = {
    "04": [-3.7122567, 40.4238823],
    "08": [-3.6823158, 40.4215533],
    "11": [-3.6773491, 40.4514734],
    "16": [-3.6392422, 40.4400457],
    "17": [-3.7133167, 40.347147],
    "18": [-3.7318356, 40.3947825],
    "24": [-3.7473445, 40.4193577],
    "27": [-3.5800258, 40.4769179],
    "35": [-3.7031662, 40.4192091],
    "36": [-3.6453104, 40.4079517],
    "38": [-3.7071303, 40.4455439],
    "39": [-3.7115364, 40.4782322],
    "40": [-3.6515286, 40.3881478],
    "47": [-3.6868138, 40.3980991],
    "48": [-3.6903729, 40.4398904],
    "49": [-3.6824999, 40.4144444],
    "50": [-3.6887449, 40.4655841],
    "54": [-3.6121394, 40.3730118],
    "55": [-3.5805649, 40.4623628],
    "56": [-3.7187679, 40.3850336],
    "57": [-3.6605173, 40.4942012],
    "58": [-3.7746101, 40.5180701],
    "59": [-3.609031, 40.465144],
    "60": [-3.6897308, 40.5005477]
}

# Create the new dictionary with the required structure
data = {
    "estacion": list(station_coordinates.keys()),
    "latitud": [item[1] for item in station_coordinates.values()],
    "longitud": [item[0] for item in station_coordinates.values()]
}

df_estaciones = pd.DataFrame(data)


Ahora, añadimos las **coordenadas métricas** $x, y$ (en metros) correspondientes al *CRS UTM (zona 30)* de cada estación:

In [4]:
df_estaciones[["x", "y"]] = df_estaciones.apply(
    lambda row: from_longlat_to_xy(row.longitud, row.latitud),
    axis=1,
    result_type="expand",
)
df_estaciones

,estacion,latitud,longitud,x,y
0,04,40.423882,-3.712257,439579.332691,4.475049e+06
1,08,40.421553,-3.682316,442117.239817,4.474771e+06
2,11,40.451473,-3.677349,442564.048969,4.478089e+06
3,16,40.440046,-3.639242,445786.171919,4.476796e+06
4,17,40.347147,-3.713317,439420.701759,4.466532e+06
5,18,40.394782,-3.731836,437891.695775,4.471833e+06
6,24,40.419358,-3.747345,436598.563517,4.474572e+06
7,27,40.476918,-3.580026,450835.202403,4.480854e+06
8,35,40.419209,-3.703166,440346.357207,4.474524e+06
9,36,40.407952,-3.645310,445245.509000,4.473237e+06


Por último, cargamos el fichero `historicoXX.csv` correspondiente a la calidad del aire medida en cada estación.

In [5]:
# Contaminante elegido: NOx.
# Si el fichero está dentro de una carpeta llamada historico, también se encontrará automáticamente.
contaminante = "NOx"
filename_calidad = f"historico{contaminante}.csv"

candidate_paths = [
    Path(filename_calidad),
    Path("historico") / filename_calidad,
    Path("datos") / filename_calidad,
]

matches = [p for p in candidate_paths if p.exists()]
if not matches:
    matches = list(Path(".").rglob(filename_calidad))

if not matches:
    raise FileNotFoundError(
        f"No se ha encontrado {filename_calidad}. Colócalo junto al notebook o dentro de la carpeta historico/."
    )

path_calidad = matches[0]
print(f"Usando fichero: {path_calidad}")

# sep=None con engine='python' permite inferir coma, punto y coma u otro separador habitual.
df_aire = pd.read_csv(path_calidad, sep=None, engine="python")
df_aire.columns = [str(c).strip() for c in df_aire.columns]

# Normalizamos el nombre de la columna temporal para poder filtrar por fechas.
possible_date_cols = [c for c in df_aire.columns if c.lower() in {"date", "fecha", "datetime"}]
if not possible_date_cols:
    raise ValueError("No se ha encontrado una columna temporal tipo 'date' o 'fecha'.")

date_col = possible_date_cols[0]
df_aire[date_col] = pd.to_datetime(df_aire[date_col])

# Rango temporal usado para resumir cada estación.
# Puedes cambiar estas fechas para estudiar otro periodo.
start_date = "2024-01-01"
end_date = "2024-12-31"

filtered_df = df_aire[(df_aire[date_col] >= start_date) & (df_aire[date_col] <= end_date)].copy()
if filtered_df.empty:
    raise ValueError(
        f"No hay datos entre {start_date} y {end_date}. Cambia el rango temporal."
    )

# Calculamos la media del contaminante por estación durante el periodo elegido.
station_columns = [c for c in filtered_df.columns if c != date_col]
average_values = filtered_df[station_columns].apply(pd.to_numeric, errors="coerce").mean()
average_values.index = average_values.index.astype(str).str.zfill(2)

print(f"Periodo analizado: {start_date} a {end_date}")
print(f"Contaminante: {contaminante}")
print("Media por estación:")
display(average_values.to_frame("media_ug_m3").T)


Usando fichero: historico\historicoNOx.csv
Periodo analizado: 2024-01-01 a 2024-12-31
Contaminante: NOx
Media por estación:


,04,08,11,16,17,18,24,27,35,36,...,48,49,50,54,55,56,57,58,59,60
media_ug_m3,38.003003,38.438066,36.408955,27.880597,55.632836,35.232628,18.807229,39.607784,36.093373,34.489552,...,27.928358,19.225225,36.307463,40.883582,30.227273,53.235821,28.40597,15.990881,26.244776,25.576577


### 0.3 Visualizador interactivo

La siguiente clase está basada en [**Plotly**](https://plotly.com/graphing-libraries/)
(principalmente su [API de visualización de mapas](https://plotly.com/python/mapbox-layers/)), <br>
y define métodos auxiliares para visualizar los resultados de la práctica. <br>

In [6]:
class MapPlotter:
    """Visualizador interactivo de mapas con Plotly"""

    CENTER = {"lat": 40.4168, "lon": -3.7038}

    def __init__(self, zoom=11, center=None, showlegend=False, **kwargs):
        self.fig = go.Figure()
        self.fig.update_layout(
            mapbox_style="open-street-map",  # o p. ej. "carto-positron",
            mapbox_zoom=zoom,
            mapbox_center=center or self.CENTER,
            margin={"r": 0, "t": 10, "l": 0, "b": 10},
            showlegend=showlegend,
            # autosize=True,
            **kwargs,
        )

    def add_longlat_points(
        self,
        lon,
        lat,
        s=15,
        color=None,
        text=None,
        edgecolor=None,
        name=None,
        adjust_center=True,
        **kwargs,
    ):
        if edgecolor is not None:
            self.fig.add_trace(
                go.Scattermapbox(
                    lat=lat,
                    lon=lon,
                    mode="markers",
                    marker_size=1.1 * s,
                    marker_color=edgecolor,
                    showlegend=False,
                    **kwargs,
                )
            )

        self.fig.add_trace(
            go.Scattermapbox(
                lat=lat,
                lon=lon,
                mode="markers",
                marker_size=s,
                marker_color=color,
                text=text,
                name=name,
                **kwargs,
            )
        )

        if adjust_center:
            self.adjust_center(lon.mean(), lat.mean())

    def add_density_heatmap(self, lon, lat, z, radius=10, adjust_center=True, **kwargs):
        self.fig.add_trace(
            go.Densitymapbox(
                lat=lat.ravel(),
                lon=lon.ravel(),
                z=z.ravel(),
                radius=radius,
                colorbar_title="μg/m³",
                **kwargs,
            )
        )

        if adjust_center:
            self.adjust_center(lon.mean(), lat.mean())

    def add_raster_heatmap(
        self,
        xgrid,
        ygrid,
        heatmap,
        cmap="Spectral_r",
        adjust_center=True,
        add_cbar=True,
        **kwargs,
    ):
        # rasterize the heatmap
        heatmap_range = np.nanmax(heatmap) - np.nanmin(heatmap)
        if heatmap_range == 0:
            heatmap_ = np.zeros_like(heatmap, dtype=float)
        else:
            heatmap_ = (heatmap - np.nanmin(heatmap)) / heatmap_range
        heatmap_ = plt.get_cmap(cmap)(heatmap_)[..., :3]
        raster_hm = Image.fromarray((heatmap_ * 255).clip(0, 255).astype(np.uint8))

        # coordenadas de las esquinas del heatmap
        bbox = np.array(
            [
                [xgrid[0, 0], ygrid[0, 0]],  # top-left
                [xgrid[0, -1], ygrid[0, -1]],  # top-right
                [xgrid[-1, -1], ygrid[-1, -1]],  # bottom-right
                [xgrid[-1, 0], ygrid[-1, 0]],  # bottom-left
            ]
        )
        lon, lat = from_xy_to_longlat(bbox[:, 0], bbox[:, 1])
        coords = np.stack((lon, lat), axis=1)

        layers = [*self.fig.layout.mapbox.layers]  # type: ignore
        layers.append(
            {
                "sourcetype": "image",
                "source": raster_hm,
                "coordinates": coords,
                "below": "traces",
                **kwargs,
            }
        )
        self.fig.layout.mapbox.layers = layers  # type: ignore

        if adjust_center:
            self.adjust_center(lon.mean(), lat.mean())

        if add_cbar:
            # add colorbar via invisible scatter trace
            self.fig.add_trace(
                go.Scattermapbox(
                    lat=[None, None],
                    lon=[None, None],
                    mode="markers",
                    marker=dict(
                        colorscale=cmap,
                        color=[np.nanmin(heatmap), np.nanmax(heatmap)],
                        cmin=np.nanmin(heatmap),
                        cmax=np.nanmax(heatmap),
                        showscale=True,
                        colorbar=dict(
                            title="μg/m³",
                            ticks="outside",
                        ),
                    ),
                )
            )

    def adjust_center(self, lon, lat):
        self.fig.update_layout(mapbox_center={"lat": lat, "lon": lon})

    def show(self):
        self.fig.show()


## 1. Kriging Geográfico

Con el setup anterior, en esta parte deberéis implementar el modelo de **Kriging** para interpolar <br>
la calidad del aire en Madrid, y visualizar los resultados obtenidos.

Para ello os proponemos utilizar, y completar en su caso, las siguientes funciones:

<div class="alert alert-danger">
&#9432; <strong>WARNING</strong> <br>
<em> Algunas estaciones no tienen todos los datos. Los archivos proporcionados son los que tienen datos. No obstante, al seleccionar las fechas podéis encontrar periodos sin datos que os dará un error.
</em>


</div>

In [7]:
def create_regular_kriging_grid(X, npts=100):
    """Malla de coordenadas sobre las que estimar la calidad del aire mediante kriging.

    Args:
        X: (..., 2) array con las coordenadas xy de las observaciones.
        npts: número de puntos a lo largo y ancho de la malla.

    Returns:
        xgrid: (npts, npts) array con las coordenadas x de la malla.
        ygrid: (npts, npts) array con las coordenadas y de la malla.
    """
    x_min, x_max = X[..., 0].min(), X[..., 0].max()
    y_min, y_max = X[..., 1].min(), X[..., 1].max()
    x, y = np.linspace(x_min, x_max, npts), np.linspace(y_min, y_max, npts)
    xgrid, ygrid = np.meshgrid(x, y)
    return xgrid, ygrid


def get_station_observations(log_data=False):
    """Une coordenadas de estaciones y medias del contaminante seleccionado."""
    df_est = df_estaciones.copy()
    df_est["estacion"] = df_est["estacion"].astype(str).str.zfill(2)
    df_est["concen."] = df_est["estacion"].map(average_values)

    # Algunas estaciones pueden no tener datos para el contaminante o periodo elegido.
    df_est = df_est.dropna(axis=0, how="any")
    if df_est.empty:
        raise ValueError("No queda ninguna estación con observaciones válidas.")

    if log_data:
        display(df_est)

    return df_est


def air_quality_kriging(df_aire, gpr, npts_grid=100, log_data=False):
    """Estimación de la calidad del aire en Madrid mediante kriging.

    Args:
        df_aire: DataFrame con los datos de calidad del aire. Se mantiene como argumento
            para respetar la estructura del notebook; las observaciones agregadas usadas
            por el GP están en la variable global average_values.
        gpr: Proceso Gaussiano a entrenar.
        npts_grid: número de puntos por dimensión de la malla de predicción.
        log_data: si True, imprime las observaciones del contaminante en cada estación.

    Returns:
        (xgrid, ygrid): coordenadas de la malla, ambas de shape (npts_grid, npts_grid).
        concentracion_pred: media estimada del contaminante en cada punto de la malla.
        sigma: desviación estándar predictiva en cada punto de la malla.
    """
    df_est = get_station_observations(log_data=log_data)

    # Datos de entrada para el proceso Gaussiano.
    X_obs = df_est[["x", "y"]].values.astype(float)
    y_obs = df_est["concen."].values.astype(float)

    # Creación de una malla regular sobre la que estimar la calidad del aire.
    xgrid, ygrid = create_regular_kriging_grid(X_obs, npts=npts_grid)

    # Entrenamiento del Proceso Gaussiano con las observaciones.
    gpr.fit(X_obs, y_obs)

    # Reorganización de la malla para ajustarla a la shape esperada por scikit-learn: (n_puntos, 2).
    X_grid = np.column_stack([xgrid.ravel(), ygrid.ravel()])

    # Predicción de la concentración media y su desviación estándar.
    concentracion_pred, sigma = gpr.predict(X_grid, return_std=True)

    # Reshape de las predicciones para ajustarlas a la malla: shape (npts_grid, npts_grid).
    concentracion_pred = concentracion_pred.reshape(xgrid.shape)
    sigma = sigma.reshape(xgrid.shape)

    return (xgrid, ygrid), concentracion_pred, sigma


def points_inside_convex_hull(X_obs, X_query):
    """Devuelve una máscara para puntos X_query que quedan dentro del área cubierta por estaciones."""
    hull = ConvexHull(X_obs)
    delaunay = Delaunay(X_obs[hull.vertices])
    return delaunay.find_simplex(X_query) >= 0


def suggest_next_station_location(xgrid, ygrid, sigma, min_distance_m=500):
    """Sugiere una ubicación para una nueva estación maximizando la incertidumbre predictiva.

    Restricciones usadas:
    - La estación debe estar dentro del casco convexo de las estaciones existentes.
    - La estación debe estar al menos a min_distance_m de cualquier estación actual,
      para evitar elegir prácticamente el mismo punto.
    """
    df_est = get_station_observations(log_data=False)
    X_obs = df_est[["x", "y"]].values.astype(float)
    X_grid = np.column_stack([xgrid.ravel(), ygrid.ravel()])

    inside = points_inside_convex_hull(X_obs, X_grid)
    distances = np.sqrt(((X_grid[:, None, :] - X_obs[None, :, :]) ** 2).sum(axis=2))
    far_enough = distances.min(axis=1) >= min_distance_m
    eligible = inside & far_enough

    if not np.any(eligible):
        raise ValueError("No hay puntos candidatos dentro del área y a la distancia mínima indicada.")

    sigma_flat = sigma.ravel()
    candidate_indices = np.where(eligible)[0]
    best_idx = candidate_indices[np.argmax(sigma_flat[eligible])]
    best_x, best_y = X_grid[best_idx]
    best_lon, best_lat = from_xy_to_longlat(best_x, best_y)

    return {
        "x": best_x,
        "y": best_y,
        "longitud": best_lon,
        "latitud": best_lat,
        "sigma": sigma_flat[best_idx],
    }


*Definición del proceso Gaussiano e interpolación de la calidad del aire:*

En primer lugar probamos con **RBF + WhiteKernel**.

Justificación del kernel:

- El RBF supone que estaciones cercanas tienden a medir valores parecidos, una hipótesis natural para interpolación espacial.
- La `length_scale` está en metros porque las coordenadas se han proyectado a UTM.
- `WhiteKernel` recoge ruido de medición y variabilidad temporal que queda al resumir todo un periodo mediante una media.
- `normalize_y=True` ayuda porque la escala absoluta de concentración depende del contaminante elegido.


In [8]:
# Kernel RBF para interpolación espacial suave.
kernel_rbf = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * RBF(length_scale=5_000.0, length_scale_bounds=(500.0, 50_000.0))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e3))
)

gpr_rbf = GaussianProcessRegressor(
    kernel=kernel_rbf,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0,
)

malla, concent_pred, concent_sigma = air_quality_kriging(
    df_aire,
    gpr_rbf,
    npts_grid=120,
    log_data=True,
)

print("Kernel RBF optimizado:")
print(gpr_rbf.kernel_)
print(f"Log-marginal-likelihood: {gpr_rbf.log_marginal_likelihood(gpr_rbf.kernel_.theta):.3f}")

fig = MapPlotter()

# Visualización de las estaciones en el mapa como puntos.
fig.add_longlat_points(
    df_estaciones["longitud"],
    df_estaciones["latitud"],
    text=df_estaciones.estacion,
    color="white",
    edgecolor="black",
)

# Visualización de la concentración predicha con kriging.
fig.add_raster_heatmap(malla[0], malla[1], concent_pred, opacity=0.6)

fig.show()


,estacion,latitud,longitud,x,y,concen.
0,04,40.423882,-3.712257,439579.332691,4.475049e+06,38.003003
1,08,40.421553,-3.682316,442117.239817,4.474771e+06,38.438066
2,11,40.451473,-3.677349,442564.048969,4.478089e+06,36.408955
3,16,40.440046,-3.639242,445786.171919,4.476796e+06,27.880597
4,17,40.347147,-3.713317,439420.701759,4.466532e+06,55.632836
5,18,40.394782,-3.731836,437891.695775,4.471833e+06,35.232628
6,24,40.419358,-3.747345,436598.563517,4.474572e+06,18.807229
7,27,40.476918,-3.580026,450835.202403,4.480854e+06,39.607784
8,35,40.419209,-3.703166,440346.357207,4.474524e+06,36.093373
9,36,40.407952,-3.645310,445245.509000,4.473237e+06,34.489552


Kernel RBF optimizado:
0.622**2 * RBF(length_scale=1.09e+04) + WhiteKernel(noise_level=0.843)
Log-marginal-likelihood: -34.051


C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:32: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:44: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:123: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(


Ahora probamos con **RationalQuadratic + WhiteKernel**.

Justificación:

- El RationalQuadratic puede interpretarse como una combinación de RBFs con distintas escalas de longitud.
- Puede ser más flexible cuando hay zonas con variaciones locales fuertes, por ejemplo estaciones cercanas a tráfico intenso, y otras zonas con gradientes más suaves.
- Si las medias anuales están muy suavizadas, es normal que produzca un resultado parecido al RBF.


In [9]:
# Kernel RationalQuadratic: permite mezclar escalas espaciales diferentes.
kernel_rq = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * RationalQuadratic(
        length_scale=5_000.0,
        alpha=1.0,
        length_scale_bounds=(500.0, 50_000.0),
        alpha_bounds=(1e-2, 1e3),
    )
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e3))
)

gpr_rq = GaussianProcessRegressor(
    kernel=kernel_rq,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0,
)

malla_rq, concent_pred_rq, concent_sigma_rq = air_quality_kriging(
    df_aire,
    gpr_rq,
    npts_grid=120,
    log_data=False,
)

print("Kernel RationalQuadratic optimizado:")
print(gpr_rq.kernel_)
print(f"Log-marginal-likelihood: {gpr_rq.log_marginal_likelihood(gpr_rq.kernel_.theta):.3f}")

fig = MapPlotter()
fig.add_longlat_points(
    df_estaciones["longitud"],
    df_estaciones["latitud"],
    text=df_estaciones.estacion,
    color="white",
    edgecolor="black",
)
fig.add_raster_heatmap(malla_rq[0], malla_rq[1], concent_pred_rq, opacity=0.6)
fig.show()


Kernel RationalQuadratic optimizado:
0.622**2 * RationalQuadratic(alpha=1e+03, length_scale=1.09e+04) + WhiteKernel(noise_level=0.843)
Log-marginal-likelihood: -34.051


c:\Users\eduar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__alpha is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:32: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:44: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:123: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scat

## Comparación y razonamiento

Los resultados de RBF y RationalQuadratic pueden ser parecidos por tres motivos:

1. Se trabaja con medias temporales, lo que reduce variaciones locales y suaviza la señal.
2. Hay pocas estaciones para estimar una superficie 2D compleja; por tanto, el prior del kernel tiene bastante peso.
3. Ambos kernels son estacionarios e isotrópicos: asumen que la correlación depende principalmente de la distancia entre puntos.

Si el `log_marginal_likelihood` del RationalQuadratic mejora claramente al RBF, lo elegiría porque permite distintas escalas espaciales. Si la diferencia es pequeña, preferiría el RBF por simplicidad e interpretabilidad.


## c) ¿Dónde colocar la siguiente estación?

Una forma razonable de usar el Proceso Gaussiano es colocar la nueva estación donde la **incertidumbre predictiva** (`sigma`) sea máxima, pero solo dentro del área ya cubierta por estaciones.

Criterio aplicado:

1. Construir el casco convexo de las estaciones actuales.
2. Considerar solo puntos de la malla que estén dentro de ese casco convexo.
3. Excluir puntos demasiado cercanos a estaciones existentes.
4. Elegir el punto con mayor desviación estándar predictiva.

Este criterio prioriza reducir incertidumbre del mapa. Si el objetivo fuera detectar máximos de contaminación, podría combinarse incertidumbre y media predicha, por ejemplo buscando zonas con alta media y alta sigma.


In [10]:
# Sugerencia de nueva estación usando la incertidumbre del modelo RBF.
# También podrías usar concent_sigma_rq si prefieres basarte en el kernel RationalQuadratic.
next_station = suggest_next_station_location(
    xgrid=malla[0],
    ygrid=malla[1],
    sigma=concent_sigma,
    min_distance_m=500,
)

print("Ubicación sugerida para la siguiente estación:")
for key, value in next_station.items():
    print(f"{key}: {value:.6f}")

fig = MapPlotter()
fig.add_longlat_points(
    df_estaciones["longitud"],
    df_estaciones["latitud"],
    text=df_estaciones.estacion,
    color="white",
    edgecolor="black",
    name="Estaciones actuales",
)
fig.add_raster_heatmap(malla[0], malla[1], concent_sigma, opacity=0.6)
fig.add_longlat_points(
    np.array([next_station["longitud"]]),
    np.array([next_station["latitud"]]),
    s=22,
    color="red",
    edgecolor="black",
    text=np.array(["Nueva estación sugerida"]),
    name="Nueva estación sugerida",
    adjust_center=False,
)
fig.show()


Ubicación sugerida para la siguiente estación:
x: 434934.659890
y: 4485388.900269
longitud: -3.768065
latitud: 40.516674
sigma: 9.263111


C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:32: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:44: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
C:\Users\eduar\AppData\Local\Temp\ipykernel_25984\2218931219.py:123: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(


## Conclusiones de la parte 2D

- Para interpolación espacial de contaminación, el RBF es una primera opción sólida porque impone suavidad con la distancia.
- El RationalQuadratic es útil cuando sospechamos que hay variaciones a distintas escalas espaciales.
- El término `WhiteKernel` es importante porque las medias por estación no son observaciones perfectas de una función espacial fija: contienen ruido de medición, variabilidad temporal y efectos locales.
- La nueva estación se puede elegir maximizando la incertidumbre predictiva dentro del casco convexo de las estaciones existentes. Así se respeta la restricción de no colocarla fuera del área cubierta y se mejora la capacidad de interpolación del sistema.

## Fuentes consultadas

- Enunciado de la práctica P8: Procesos Gaussianos.
- Documentación de `GaussianProcessRegressor` y kernels de `scikit-learn`.
- Documentación de PyProj para transformación de coordenadas.
- Portal de datos abiertos del Ayuntamiento de Madrid para el origen de los datos de calidad del aire.
